<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/cursor%2Fcolab-llm-diffusion-drive-e32d/llm_hub_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab inference hub: LLMs + diffusion on Google AI Pro GPUs

One authenticated OpenAI-compatible API on a Colab GPU for **chat** (llama.cpp / vLLM) and **text-to-image** (Diffusers).
Built for **Google AI Pro / Colab Pro** runtimes (L4, A100, G4): models download once to **Google Drive**, then restore into later sessions.

- **Models live on Drive** at `MyDrive/llm_models/models/<id>` (engines and wheels under `builds/`).
- **Chat:** `POST /v1/chat/completions` — name any catalog LLM; the hub loads or switches engines on demand.
- **Images:** `POST /v1/images/generations` — SD 1.5, SDXL, SDXL Turbo, FLUX.1 Schnell (whatever fits this GPU).
- **Your harness** connects with a base URL + API key (stable key kept on Drive). The last cell prints a dsh provider block for chat models.

Hub code is fetched from this repo each run. Runtime → Change runtime type → **L4, A100 or G4 GPU** (AI Pro / Colab Pro), then run the cells in order.


## 1. Mount Drive and fetch the hub code

In [ ]:
import os, re, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

HUB_REPO  = "https://github.com/Biegle-W/google_colab_llm"
HUB_REF   = "cursor/colab-llm-diffusion-drive-e32d"  # pin until merged; then set None for default branch
DRIVE_DIR = "/content/drive/MyDrive/llm_models"
HUB_DIR   = "/content/hub_repo"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{cmd}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

sh(f"rm -rf {HUB_DIR}")
if HUB_REF and re.fullmatch(r"[0-9a-f]{40}", HUB_REF):          # a commit SHA has to be fetched by hash
    sh(f"mkdir -p {HUB_DIR} && cd {HUB_DIR} && git init -q && git fetch -q --depth 1 {HUB_REPO} {HUB_REF} && git checkout -q FETCH_HEAD")
else:
    sh(f"git clone -q --depth 1 {'--branch ' + HUB_REF if HUB_REF else ''} {HUB_REPO} {HUB_DIR}")
sys.path.insert(0, HUB_DIR)
import colab_llm_hub
print("hub", colab_llm_hub.__version__, "| commit", sh(f"cd {HUB_DIR} && git rev-parse --short HEAD"))


## 2. Detect the GPU and see which models fit it

In [ ]:
from colab_llm_hub import catalog, gpu

G = gpu.detect()[0]
VRAM = G["vram_gb"]
ENTRIES = catalog.load()
print(f"{G['name']}: {VRAM} GB VRAM, compute capability {G['compute_cap']} (sm_{G['arch']})")
!free -h | head -2
!df -h /content | tail -1
print()
print(f"{'model id':32s} {'kind':5s} {'engine':10s} {'size':>7s}  {'ctx/edge':>12s}")
for e in ENTRIES:
    fit = catalog.fits(e, VRAM)
    kind = catalog.kind_for(e)
    label = "edge" if kind == "image" else "ctx"
    ctx = f"{catalog.context_for(e, VRAM):,} {label}" if fit else f"needs {catalog.min_vram_gb(e)} GB"
    print(f"{e['id']:32s} {kind:5s} {e['backend']:10s} {e['size_gb']:6.1f}G  {ctx:>12s}" + ("" if fit else "   (does not fit)"))
if not catalog.visible(ENTRIES, VRAM):
    print("\nNo catalog model fits this GPU. Use an L4, A100 or G4 runtime (AI Pro / Colab Pro).")


## 3. Config

For chat models, catalog **context** values are estimates. For image models, the second column is the **max edge in pixels** for this VRAM. Override either with `CTX_OVERRIDE`.


In [ ]:
# Models to download to Drive now. The hub also fetches any listed model the first time a request names it,
# but a big download mid-request is slow — prefetch what you use.
PREFETCH = ["qwen3.8-27b-q4", "sdxl-turbo"]   # chat + image examples; trim to what fits this GPU

ENGINES = ["llama", "diffusers"]   # add "vllm" for FP8 safetensors LLMs (several GB, cached on Drive)
CTX_OVERRIDE = {}                  # e.g. {"qwen3.8-27b-q4": 32768, "sdxl-turbo": 768}

LLAMA_CPP_REF = "4c3c5810f87efe177949f1b85ae509d38a2a311f"
REBUILD_LLAMA = False
REFRESH_WHEELS = False             # True = re-download vLLM / Diffusers wheels to Drive

TUNNEL = "quick"                   # "quick" | "named" (see Notes)
PORT = 8000

unknown = [m for m in PREFETCH if m not in {e["id"] for e in ENTRIES}]
assert not unknown, f"unknown model ids: {unknown}"
not_fit = [m for m in PREFETCH if not catalog.fits(catalog.by_id(ENTRIES)[m], VRAM)]
assert not not_fit, f"these do not fit this GPU: {not_fit}"
needed = {catalog.by_id(ENTRIES)[m]["backend"] for m in PREFETCH}
assert needed <= set(ENGINES), f"PREFETCH needs engines {sorted(needed - set(ENGINES))}; add them to ENGINES"


## 4. Install the engines (built or downloaded once, then restored from Drive)

In [ ]:
from colab_llm_hub import install

BUILDS = f"{DRIVE_DIR}/builds"
LLAMA_BIN = "llama-server"
if "llama" in ENGINES:
    LLAMA_BIN = install.ensure_llama(BUILDS, G["arch"], ref=LLAMA_CPP_REF, rebuild=REBUILD_LLAMA)
    print("llama-server:", LLAMA_BIN)
if "vllm" in ENGINES:
    print("vLLM", install.ensure_vllm(BUILDS, refresh=REFRESH_WHEELS))
if "diffusers" in ENGINES:
    print("diffusers", install.ensure_diffusers(BUILDS, refresh=REFRESH_WHEELS))


## 5. Download the models to Google Drive (skipped for anything already saved)

In [ ]:
%pip install -q -U huggingface_hub hf_xet
import threading, time
from colab_llm_hub.store import ModelStore

try:                                    # a Hugging Face token (Colab secret HF_TOKEN) is only needed for gated models
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

store = ModelStore(DRIVE_DIR, "/content/models")
for mid in PREFETCH:
    e = catalog.by_id(ENTRIES)[mid]
    if store.state(e)["on_drive"]:
        print(f"{mid}: already on Drive")
        continue
    stop = threading.Event()
    def report():
        while not stop.wait(15):
            p = store.progress.get(mid, {})
            print(f"  {mid}: {p.get('phase', '')} {p.get('done_bytes', 0) / 1e9:.1f} / {e['size_gb']:.1f} GB")
    threading.Thread(target=report, daemon=True).start()
    t0 = time.time()
    try:
        store.prefetch(e)
    finally:
        stop.set()
    print(f"{mid}: saved to Drive in {(time.time() - t0) / 60:.1f} min")

## 6. Start the hub and open the tunnel

Re-running this cell restarts the hub. The API key is created once and kept on Drive, so your harness settings stay valid across sessions.

In [ ]:
import json, requests, shutil, time
from colab_llm_hub import keyfile, tunnel, provider

subprocess.run("pkill -TERM -f '[c]olab_llm_hub serve'; pkill -f '[c]loudflared'", shell=True)   # stop a previous run of this cell ([c] stops the pattern matching its own shell)
time.sleep(3)

API_KEY = keyfile.load_or_create(f"{DRIVE_DIR}/hub/api_key")
args = [sys.executable, "-m", "colab_llm_hub", "serve", "--drive-dir", DRIVE_DIR, "--port", str(PORT),
        "--llama-bin", LLAMA_BIN, "--engines", ",".join(ENGINES)]
for k, v in CTX_OVERRIDE.items():
    args += ["--ctx", f"{k}={v}"]
hub_proc = subprocess.Popen(args, cwd=HUB_DIR, env=dict(os.environ, PYTHONPATH=HUB_DIR),
                            stdout=open("/content/hub.log", "w"), stderr=subprocess.STDOUT)

AUTH = {"Authorization": f"Bearer {API_KEY}"}
for _ in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", headers=AUTH, timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    if hub_proc.poll() is not None:
        raise RuntimeError("The hub exited:\n" + open("/content/hub.log").read()[-3000:])
    time.sleep(1)
else:
    raise RuntimeError("The hub did not start:\n" + open("/content/hub.log").read()[-3000:])

cf = tunnel.ensure_cloudflared()
if TUNNEL == "named":
    from google.colab import userdata
    tunnel_proc = tunnel.start_named(userdata.get("CLOUDFLARE_TUNNEL_TOKEN"))
    BASE = "https://" + userdata.get("CLOUDFLARE_HOSTNAME").removeprefix("https://").strip("/")
else:
    tunnel_proc, BASE = tunnel.start_quick(PORT)

MODELS = requests.get(f"http://127.0.0.1:{PORT}/v1/models", headers=AUTH).json()["data"]
json.dump({"base_url": BASE + "/v1", "updated": time.strftime("%Y-%m-%d %H:%M:%S"), "gpu": G["name"],
           "models": [m["id"] for m in MODELS]}, open(f"{DRIVE_DIR}/hub/endpoint.json", "w"), indent=1)

print("Base URL :", BASE + "/v1")
print("API key  :", API_KEY)
print(f"Models   : {', '.join(m['id'] for m in MODELS)}")
print()
print("Any OpenAI-compatible client:")
print(provider.openai_env(BASE, API_KEY))
print("dsh: on your computer, in a clone of this repo, this adds the provider to its settings for you:")
print(f"  python -m colab_llm_hub.configure_dsh --url {BASE} --key {API_KEY} --settings <path to dsh settings.yaml>")
print("or merge this block by hand, and set the key in the environment where dsh runs:")
print(f"  export COLAB_LLM_API_KEY={API_KEY}\n")
print(provider.dsh_provider_yaml(BASE, MODELS))

## 7. Try chat and image generation

The first request to a model loads it (download or Drive→local copy, then GPU load). Streaming chat waits with keep-alives; non-streaming may return `503` + `Retry-After` while loading — re-run the cell.


In [ ]:
from IPython.display import display, Image as IPyImage
import base64

chat_models = [m for m in MODELS if m.get("kind") != "image"]
image_models = [m for m in MODELS if m.get("kind") == "image"]

if chat_models:
    MODEL = next((m for m in PREFETCH if m in {x["id"] for x in chat_models}), chat_models[0]["id"])
    t0 = time.time()
    r = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", headers=AUTH, timeout=900,
                      json={"model": MODEL, "max_tokens": 200,
                            "messages": [{"role": "user", "content": "In two sentences, what is a model hub?"}]})
    if r.status_code == 503 and r.json().get("error", {}).get("code") == "model_loading":
        print("Chat model still loading; re-run this cell in a minute.")
    else:
        j = r.json()
        print("=== chat:", MODEL, "===")
        print(j["choices"][0]["message"]["content"] if r.ok else j)
        print(f"[{time.time() - t0:.1f} s]\n")

if image_models:
    IMG = next((m for m in PREFETCH if m in {x["id"] for x in image_models}), image_models[0]["id"])
    t0 = time.time()
    r = requests.post(f"http://127.0.0.1:{PORT}/v1/images/generations", headers=AUTH, timeout=900,
                      json={"model": IMG, "prompt": "a watercolor fox under pine trees, soft light",
                            "size": "768x768", "n": 1})
    if r.status_code == 503 and r.json().get("error", {}).get("code") == "model_loading":
        print("Image model still loading; re-run this cell in a minute.")
    else:
        j = r.json()
        print("=== image:", IMG, j.get("size"), "===")
        if r.ok:
            display(IPyImage(data=base64.b64decode(j["data"][0]["b64_json"])))
            print(f"[{time.time() - t0:.1f} s]")
        else:
            print(j)


## 8. Status and logs (re-run any time)

In [ ]:
st = requests.get(f"http://127.0.0.1:{PORT}/v1/hub/status", headers=AUTH).json()
print(json.dumps({k: st[k] for k in ("gpu", "vram_gb", "active", "loading", "phase", "inflight", "errors", "local_free_gb")}, indent=1))
for mid, p in st["downloads"].items():
    print(f"  {mid}: {p.get('phase')} {p.get('done_bytes', 0) / 1e9:.1f} / {p.get('total_bytes', 0) / 1e9:.1f} GB")
print("\n--- hub log ---"); print(open("/content/hub.log").read()[-1500:])
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 9. Stop

In [ ]:
hub_proc.terminate()        # stops the hub and the engine it started
tunnel_proc.terminate()

## Notes

- **Google AI Pro / Colab Pro:** pick Runtime → Change runtime type → L4, A100 or G4. T4 is too small for most catalog LLMs (use `llama_cpp_t4.ipynb`).
- **Drive layout:** `MyDrive/llm_models/models/<id>` (weights), `builds/` (llama.cpp tarball, vLLM/Diffusers wheels), `hub/api_key` + `hub/endpoint.json`.
- **Chat:** point any OpenAI-compatible client at the Base URL with the API key; name a model from `/v1/models` with `kind: chat`.
- **Images:** `POST /v1/images/generations` with `{"model": "sdxl-turbo", "prompt": "...", "size": "1024x1024"}`. Response includes `data[].b64_json`.
- **One model at a time.** Switching waits for in-flight replies, then swaps engines (chat ↔ image included).
- **Gated HF repos** (some Stability / BFL models): add Colab secret `HF_TOKEN` after accepting the model license on Hugging Face.
- **Stable URL:** set `TUNNEL = "named"` with secrets `CLOUDFLARE_TUNNEL_TOKEN` and `CLOUDFLARE_HOSTNAME`.
- **Security:** the tunnel URL is public; only the API key protects GPU use. Rotate by deleting `hub/api_key` on Drive.
- Add models in `colab_llm_hub/catalog.json` (GGUF / safetensors / Diffusers repos). Image `contexts` are `[[min_vram_gb, max_edge_px], ...]`.
- Logs: `/content/hub.log`, `/content/hub_logs/`, `/content/install.log`.
